# Speed comparison

How fast fse trains depends mostly on three choices: the input class, the number of workers and the storage format of the vectors. This notebook times `train` end to end for each of them on one machine. The numbers vary with the machine and its load; for careful measurements in fresh processes, with the load average recorded, use `benchmarks/bench_train.py` in the repository.

In [1]:
import csv
import gzip
import logging
import os
import platform
import sys
import tempfile
import time
import warnings

warnings.filterwarnings("ignore", message="IProgress not found")

import gensim.downloader as api
import pandas as pd

from fse import SIF, Average, CorpusFile, IndexedList, TextList, Vectors
from fse.models.average import FAST_VERSION

logging.basicConfig(level=logging.WARNING)
print(platform.platform(), platform.processor(), f"{os.cpu_count()} logical CPUs")
print("compiled routines:", FAST_VERSION == 1)
print("load average:", os.getloadavg())

macOS-27.0.1-arm64-arm-64bit-Mach-O arm 10 logical CPUs
compiled routines: True
load average: (5.25537109375, 6.6357421875, 5.8798828125)


The data are the non-empty Quora duplicate questions from gensim-data, repeated four times, and the vectors are `glove-wiki-gigaword-100`.

In [2]:
path = api.load("quora-duplicate-questions", return_path=True)  # its own loader needs an old smart_open
csv.field_size_limit(sys.maxsize)
with gzip.open(path, "rt", encoding="utf-8") as f:
    data = list(csv.DictReader(f, delimiter="\t"))
questions = [q for d in data for q in (d["question1"], d["question2"]) if q.split()] * 4
glove = Vectors.from_pretrained("glove-wiki-gigaword-100")
print(f"{len(questions):,} sentences")

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Fetching 4 files: 100%|██████████| 4/4 [00:00<00:00, 3089.16it/s]

3,234,312 sentences


In [3]:
def sentences_per_second(model, sentences, repeat=3):
    """Best of `repeat` trainings, in sentences per second."""
    best = float("inf")
    for _ in range(repeat):
        start = time.perf_counter()
        model.train(sentences)
        best = min(best, time.perf_counter() - start)
    return len(questions) / best

## Input classes

`IndexedList` needs the sentences split into Python lists first, `TextList` splits the strings in compiled code, and `CorpusFile` reads them from a file. With several workers, all three are read by every worker for its own part of the input.

In [4]:
split = IndexedList([q.split() for q in questions])
text = TextList(questions)
tmp = tempfile.NamedTemporaryFile("w", suffix=".txt", delete=False, encoding="utf-8")
tmp.write("".join(q.replace("\n", " ") + "\n" for q in questions))
tmp.close()
corpus = CorpusFile(tmp.name)

inputs = {"IndexedList": split, "TextList": text, "CorpusFile": corpus}
workers = sorted({1, 2, 4, os.cpu_count()})
rows = []
for name, sentences in inputs.items():
    for w in workers:
        rows.append({"input": name, "workers": w, "sentences/s": sentences_per_second(Average(glove, workers=w), sentences)})
pd.DataFrame(rows).pivot(index="input", columns="workers", values="sentences/s").map(lambda v: f"{v:,.0f}")

workers,1,2,4,10
input,,,,
CorpusFile,"1,276,715","2,550,659","4,910,131","8,836,654"
IndexedList,"963,026","1,864,200","2,677,223","2,526,830"
TextList,"1,301,688","2,766,168","5,336,904","9,697,435"


## Models

SIF also learns and removes principal components, which costs a second pass over the sentence vectors.

In [5]:
rows = []
for name, cls, kwargs in [("Average", Average, {}), ("SIF", SIF, {"lang_freq": "en"})]:
    for w in workers:
        rows.append({"model": name, "workers": w, "sentences/s": sentences_per_second(cls(glove, workers=w, **kwargs), text)})
pd.DataFrame(rows).pivot(index="model", columns="workers", values="sentences/s").map(lambda v: f"{v:,.0f}")

workers,1,2,4,10
model,,,,
Average,"1,331,552","2,770,636","5,210,924","9,431,216"
SIF,"1,133,025","2,165,314","4,333,319","7,240,959"


## Storage formats

float16 and int8 vectors take half or a quarter of the memory. `wv_dtype` sets the format of the word vectors that are averaged, `sv_dtype` the format in which the sentence vectors are stored; the table shows what each combination costs or saves in time on this machine, with all workers.

In [6]:
rows = []
w = os.cpu_count()
for wv_dtype in ["float32", "float16", "int8"]:
    for sv_dtype in ["float32", "int8"]:
        model = Average(glove, workers=w, wv_dtype=wv_dtype, sv_dtype=sv_dtype)
        rows.append({"wv_dtype": wv_dtype, "sv_dtype": sv_dtype, "sentences/s": sentences_per_second(model, text)})
pd.DataFrame(rows).pivot(index="wv_dtype", columns="sv_dtype", values="sentences/s").map(lambda v: f"{v:,.0f}")

sv_dtype,float32,int8
wv_dtype,,
float16,"9,246,547","3,961,231"
float32,"9,103,203","3,229,441"
int8,"9,878,406","3,563,625"


In [7]:
os.unlink(tmp.name)